# 02 — Frequency Model

Frequency model гледа колко често се появява всяко число в историческите данни. Това е статистически сигнал, не гаранция.


In [ ]:
from pathlib import Path
from itertools import combinations
from collections import Counter

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

pd.set_option("display.max_columns", 80)
pd.set_option("display.width", 160)

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "data").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

DATA_DIR = PROJECT_ROOT / "data"
REPORTS_DIR = PROJECT_ROOT / "reports"
MODELS_DIR = PROJECT_ROOT / "models"


def read_csv_safe(path, **kwargs):
    path = Path(path)
    if not path.exists():
        print(f"Missing file: {path}")
        return pd.DataFrame()
    return pd.read_csv(path, **kwargs)


def number_columns(df: pd.DataFrame) -> list[str]:
    return [col for col in ["n1", "n2", "n3", "n4", "n5", "n6"] if col in df.columns]


def parse_numbers_text(value) -> list[int]:
    if pd.isna(value):
        return []
    parts = str(value).replace(";", ",").replace("|", ",").split(",")
    nums = []
    for part in parts:
        part = part.strip()
        if part.isdigit():
            nums.append(int(part))
    return nums


def extract_ticket_numbers(df: pd.DataFrame) -> pd.DataFrame:
    if df.empty:
        return df
    if len(number_columns(df)) == 6:
        return df.copy()
    text_col = next((c for c in df.columns if "numbers" in c.lower() or "числа" in c.lower()), None)
    if text_col is None:
        return df.copy()
    out = df.copy()
    parsed = out[text_col].apply(parse_numbers_text)
    for i in range(6):
        out[f"n{i+1}"] = parsed.apply(lambda nums: nums[i] if len(nums) > i else np.nan)
    return out


def show_latest_draw(df: pd.DataFrame) -> None:
    if df.empty:
        print("No draw data loaded.")
        return
    latest = df.tail(1).iloc[0]
    nums = [int(latest[col]) for col in number_columns(df)]
    print("Rows:", len(df))
    print("Latest date:", latest.get("date", "n/a"))
    print("Draw number:", latest.get("draw_number", latest.get("draw_no", "n/a")))
    print("Numbers:", nums)


def file_status(paths):
    rows = []
    for path in paths:
        path = Path(path)
        rows.append({
            "file": str(path.relative_to(PROJECT_ROOT)) if str(path).startswith(str(PROJECT_ROOT)) else str(path),
            "exists": path.exists(),
            "size_kb": round(path.stat().st_size / 1024, 2) if path.exists() else 0,
        })
    return pd.DataFrame(rows)


draws = read_csv_safe(DATA_DIR / "historical_draws.csv")
normalized = read_csv_safe(DATA_DIR / "v40_normalized_draw_events.csv")
canonical = read_csv_safe(DATA_DIR / "v41_canonical_draw_events.csv")
show_latest_draw(draws)


## Frequency table


In [ ]:
num_cols = number_columns(draws)
all_numbers = draws[num_cols].to_numpy().ravel()
frequency = (
    pd.Series(all_numbers)
    .value_counts()
    .reindex(range(1, 50), fill_value=0)
    .sort_index()
    .rename_axis("number")
    .reset_index(name="frequency")
)
frequency["frequency_share"] = frequency["frequency"] / frequency["frequency"].sum()
frequency["rank_desc"] = frequency["frequency"].rank(ascending=False, method="dense").astype(int)
frequency.sort_values("frequency", ascending=False).head(15)


## Top и cold numbers


In [ ]:
top_numbers = frequency.sort_values("frequency", ascending=False).head(10)
cold_numbers = frequency.sort_values("frequency", ascending=True).head(10)
display(top_numbers)
display(cold_numbers)
ax = top_numbers.set_index("number")["frequency"].plot(kind="bar", figsize=(10, 4), title="Top 10 most frequent numbers")
ax.set_xlabel("Number")
ax.set_ylabel("Frequency")
plt.show()
ax = cold_numbers.set_index("number")["frequency"].plot(kind="bar", figsize=(10, 4), title="Top 10 coldest numbers")
ax.set_xlabel("Number")
ax.set_ylabel("Frequency")
plt.show()


## Rolling frequency за топ числа


In [ ]:
selected_numbers = top_numbers["number"].head(5).tolist()
window = 200
binary = pd.DataFrame(index=draws.index)
for number in selected_numbers:
    binary[number] = draws[num_cols].eq(number).any(axis=1).astype(int)
rolling = binary.rolling(window=window, min_periods=20).mean()
ax = rolling.plot(figsize=(12, 5), title=f"Rolling appearance rate, window={window}")
ax.set_xlabel("Draw index")
ax.set_ylabel("Rolling rate")
plt.show()
